# Model Comparison — Twitter Fake Account Detection

Compare SLP, Logistic Regression, Random Forest, and XGBoost.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, roc_auc_score
)
from sklearn.model_selection import cross_val_score, cross_val_predict

import sys
sys.path.insert(0, "..")
from src.preprocessing import fit_preprocessor, transform_preprocessor
from src.features import add_all_features
from src.scaling import fit_scaler, transform_scaler, combine_scaled_features, FEATURE_COLUMNS
from src.model import prepare_train_data, prepare_test_data
from src.pipeline import build_model, MODEL_REGISTRY
from src.config import load_config

plt.rcParams["figure.dpi"] = 120
sns.set_style("whitegrid")

config = load_config()
DATA_CFG = config["data"]
TRAINING_CFG = config.get("training", {})
CV_FOLDS = TRAINING_CFG.get("cv_folds", 5)
print(f"CV folds: {CV_FOLDS}")

In [ ]:
# Load and preprocess
print("Loading data...")
df_train = pd.read_excel(DATA_CFG["train_path"])
df_test = pd.read_excel(DATA_CFG["test_path"])
ref_date = DATA_CFG.get("reference_date", "2025-12-02")

df_train_proc, lang_mapping = fit_preprocessor(df_train, ref_date)
df_train_proc = add_all_features(df_train_proc)
scaler, X_train_scaled = fit_scaler(df_train_proc)
df_train_final = combine_scaled_features(df_train_proc, X_train_scaled)

df_test_proc = transform_preprocessor(df_test, lang_mapping, ref_date)
df_test_proc = add_all_features(df_test_proc)
X_test_scaled = transform_scaler(df_test_proc, scaler)
df_test_final = combine_scaled_features(df_test_proc, X_test_scaled)

X_train, y_train = prepare_train_data(df_train_final)
X_test, y_test = prepare_test_data(df_test_final)
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

---
## 1. Training & Cross-Validation Comparison

In [ ]:
results = []
models = {}

for name in MODEL_REGISTRY:
    print(f"\n{'='*40}")
    print(f"Training {name}...")
    
    model = build_model(name, config.get("model", {}))
    
    # Training time
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0
    
    # CV scores
    t0 = time.time()
    cv_scores = cross_val_score(model, X_train, y_train, cv=CV_FOLDS, scoring="accuracy")
    cv_time = time.time() - t0
    
    # Test predictions
    y_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    test_prec = precision_score(y_test, y_pred, zero_division=0)
    test_rec = recall_score(y_test, y_pred, zero_division=0)
    test_f1 = f1_score(y_test, y_pred, zero_division=0)
    
    # ROC AUC
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        y_proba = None
        roc_auc = None
    
    models[name] = {
        "model": model,
        "cv_scores": cv_scores,
        "y_pred": y_pred,
        "y_proba": y_proba,
        "confusion_matrix": confusion_matrix(y_test, y_pred),
        "train_time": train_time,
        "cv_time": cv_time,
    }
    
    results.append({
        "model": name,
        "CV mean (acc)": cv_scores.mean().round(4),
        "CV std (acc)": cv_scores.std().round(4),
        "Test accuracy": round(test_acc, 4),
        "Precision": round(test_prec, 4),
        "Recall": round(test_rec, 4),
        "F1-score": round(test_f1, 4),
        "ROC AUC": round(roc_auc, 4) if roc_auc else "N/A",
        "Train time (s)": round(train_time, 3),
        "CV time (s)": round(cv_time, 3),
    })
    
    print(f"  CV acc: {cv_scores.mean():.4f} (+- {cv_scores.std():.4f})")
    print(f"  Test acc: {test_acc:.4f}, F1: {test_f1:.4f}")
    print(f"  Train time: {train_time:.3f}s, CV time: {cv_time:.3f}s")

In [ ]:
comparison_df = pd.DataFrame(results)
print("=== Model Comparison ===")
display(comparison_df.style.highlight_max(color="#2ecc71", axis=0, subset=["CV mean (acc)", "Test accuracy", "Precision", "Recall", "F1-score", "ROC AUC"]))

---
## 2. Cross-Validation Scores (per fold)

In [ ]:
cv_data = []
for name in MODEL_REGISTRY:
    for fold, score in enumerate(models[name]["cv_scores"], 1):
        cv_data.append({"model": name, "fold": fold, "accuracy": score})
cv_df = pd.DataFrame(cv_data)

plt.figure(figsize=(10, 5))
sns.boxplot(data=cv_df, x="model", y="accuracy", palette="Set2")
sns.stripplot(data=cv_df, x="model", y="accuracy", color="black", size=6, jitter=True)
plt.title(f"Cross-Validation Accuracy ({CV_FOLDS}-fold)")
plt.ylabel("Accuracy")
plt.xlabel("Model")
plt.tight_layout()
plt.show()

---
## 3. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for idx, name in enumerate(MODEL_REGISTRY):
    row, col = divmod(idx, 2)
    cm = models[name]["confusion_matrix"]
    
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Real", "Fake"],
                yticklabels=["Real", "Fake"],
                ax=axes[row, col], cbar=False)
    axes[row, col].set_title(f"{name}")
    axes[row, col].set_xlabel("Predicted")
    axes[row, col].set_ylabel("Actual")

plt.suptitle("Confusion Matrices — Test Set", fontsize=14)
plt.tight_layout()
plt.show()

---
## 4. ROC Curves Overlay

In [ ]:
plt.figure(figsize=(9, 7))
colors = ["#3498db", "#e67e22", "#2ecc71", "#e74c3c"]

for idx, name in enumerate(MODEL_REGISTRY):
    y_proba = models[name]["y_proba"]
    if y_proba is not None:
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=colors[idx], lw=2,
                 label=f"{name} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Test Set")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Feature Importance Comparison

In [ ]:
def extract_feature_importance(model, feature_names, model_name):
    if hasattr(model, "coefs_") and model.coefs_[0].ndim >= 2:
        # SLP / MLP
        weights = model.coefs_[0]
        if weights.ndim == 2 and weights.shape[1] == 1:
            w = weights[:, 0]
        else:
            w = abs(weights).mean(axis=1)
        return pd.DataFrame({"feature": feature_names, model_name: abs(w)})
    elif hasattr(model, "coef_"):
        # Logistic Regression
        w = model.coef_[0]
        return pd.DataFrame({"feature": feature_names, model_name: abs(w)})
    elif hasattr(model, "feature_importances_"):
        # RF / XGB
        return pd.DataFrame({"feature": feature_names, model_name: model.feature_importances_})
    else:
        return pd.DataFrame({"feature": feature_names, model_name: 0})


imp_dfs = []
for name in MODEL_REGISTRY:
    imp_dfs.append(
        extract_feature_importance(models[name]["model"], FEATURE_COLUMNS, name)
    )

# Merge all into one table
imp_merged = imp_dfs[0]
for df in imp_dfs[1:]:
    imp_merged = imp_merged.merge(df, on="feature")

display(imp_merged.round(4))

In [ ]:
# Normalize each column for visual comparison
imp_norm = imp_merged.copy()
for col in MODEL_REGISTRY:
    total = imp_norm[col].sum()
    if total > 0:
        imp_norm[col] = imp_norm[col] / total

imp_norm = imp_norm.set_index("feature")

fig, ax = plt.subplots(figsize=(10, 6))
imp_norm.plot(kind="barh", ax=ax, width=0.8)
ax.set_title("Feature Importance Comparison (normalized per model)")
ax.set_xlabel("Relative Importance")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

---
## 6. Training Time Benchmark

In [ ]:
time_data = []
for r in results:
    time_data.append({"model": r["model"], "time": r["Train time (s)"], "phase": "Train"})
    time_data.append({"model": r["model"], "time": r["CV time (s)"], "phase": "CV"})
time_df = pd.DataFrame(time_data)

plt.figure(figsize=(9, 4))
sns.barplot(data=time_df, x="model", y="time", hue="phase", palette="Set2")
plt.title("Training & CV Time Benchmark")
plt.ylabel("Time (seconds)")
plt.xlabel("Model")
plt.legend()
plt.tight_layout()
plt.show()

---
## 7. Final Recommendation

In [ ]:
# Pick the best model by F1-score (handles class imbalance better than accuracy)
best_idx = np.argmax([r["F1-score"] for r in results])
best = results[best_idx]

print("=" * 55)
print("  FINAL RECOMMENDATION")
print("=" * 55)
print(f"\nBest model: {best['model']}")
print(f"  CV accuracy (mean): {best['CV mean (acc)']:.4f}")
print(f"  Test accuracy:      {best['Test accuracy']:.4f}")
print(f"  Precision:          {best['Precision']:.4f}")
print(f"  Recall:             {best['Recall']:.4f}")
print(f"  F1-score:           {best['F1-score']:.4f}")
print(f"  ROC AUC:            {best['ROC AUC']}")
print(f"  Train time:         {best['Train time (s)']:.3f}s")

print(f"\nSummary:\n")

for r in sorted(results, key=lambda x: x["F1-score"], reverse=True):
    star = " ⬅ BEST" if r["model"] == best["model"] else ""
    print(f"  {r['model']:25s} | F1: {r['F1-score']:.4f} | Acc: {r['Test accuracy']:.4f} | AUC: {r['ROC AUC']} | Time: {r['Train time (s)']:.3f}s{star}")

### Decision

- If **F1-score** is the priority (balancing precision & recall), use the model highlighted above.
- If **interpretability** matters, Logistic Regression provides clear coefficients.
- If **speed** is critical, SLP trains the fastest.
- For **maximum performance**, Random Forest or XGBoost typically excel with enough data.